# Project 07: Burrows-Wheeler Transform

**Note**: Keep the cell below as is. It is for formatting purposes

In [31]:
%%html
<style>
.MathJax_Display,  /* MathJax 2 */
mjx-container {    /* MathJax 3 */
  text-align: left !important;
}
</style>

## Learning Objectives

1. Students will be able to explain the fundamental principles of the Burrows-Wheeler Transform algorithm, including its purpose, process, and applications in data compression and bioinformatics.
2. Students will be able to implement the Burrows-Wheeler Transform using both the naive approach and the more efficient suffix array method.
3. Students will be able to construct auxiliary data structures (count array and occurrence array) needed for efficient pattern matching with the FM-Index.
4. Students will be able to apply the BWT-based pattern matching algorithm to find exact string matches within a reference text.

---

## Background
The Burrows-Wheeler Transform (BWT) is a powerful algorithm used in data compression and bioinformatics. Developed by Michael Burrows and David Wheeler in 1994, it rearranges a character string into runs of similar characters, making it more amenable to compression techniques like move-to-front encoding and run-length encoding. Despite adding a small amount of overhead, BWT's ability to group similar characters together significantly improves compression efficiency.

The BWT works by sorting all possible rotations of a string lexicographically and then extracting the last column of the sorted matrix. What makes BWT remarkable is that this transformation is reversible without storing much additional information - only the position of the original string in the sorted rotations is needed.

In recent years, BWT has found extensive applications beyond compression. It's particularly valuable in bioinformatics for efficient sequence alignment of DNA reads against reference genomes, as implemented in popular tools like Bowtie and BWA. The algorithm's efficiency in handling large datasets has made it indispensable for processing the massive amounts of data generated by next-generation sequencing technologies.

## The BWT Algorithm

![](assets/rotation.png)

> Credit: Ferragina, P., Giancarlo, R., Manzini, G., & Sciortino, M. (2005). Boosting textual compression in optimal linear time. In Journal of the ACM (Vol. 52, Issue 4, pp. 688–713). Association for Computing Machinery (ACM). https://doi.org/10.1145/1082036.1082043 

At its core, the BWT algorithm involves creating all cyclic rotations of an input string, sorting them lexicographically, and then extracting the last column.

Let's define the pseudocode for our BWT function:

$$
{\small
\begin{array}{l}
\textbf{function } \text{BWT}(text) \\
\quad \text{Append a unique end-of-string character to } text \text{ if not present} \\
\quad \text{Create a list to store all possible rotations of the text} \\
\quad \textbf{for each } \text{possible starting position in the text} \textbf{ do} \\
\quad\quad \text{Generate a rotation starting at this position and add to the list} \\
\quad \text{Sort all rotations lexicographically} \\
\quad \text{Initialize an empty string for the transformed result} \\
\quad \textbf{for each } \text{rotation in the sorted list} \textbf{ do} \\
\quad\quad \text{Append the last character of this rotation to the transformed result} \\
\quad \textbf{return } \text{transformed result and position of original text in sorted rotations}
\end{array}
}
$$

In [35]:
def BWT(string: str) -> str:
    """Function to calculate Burrows-Wheeler Transform for a given string.
    
    Computes the Burrows-Wheeler Transform by creating all rotations of the input
    string, sorting them lexicographically, and extracting the last column.
    
    Args:
        string: The input string to transform.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        Position in sorted rotations
        
    Examples:
        >>> BWT('googol')
        'lo$oogg'
        
        >>> BWT('banana$')
        'annb$aa'

    Pseudocode:
    IF string does not end with $:
        append $ to string
    FOR i from 0 to length of string - 1:
        build rotation = string[i:] + string[:i]     (move the first i chars to the back)
        store the pair (rotation, i)
    SORT the pairs alphabetically by rotation, keeping each i with its row
    FOR each (rotation, i) in sorted order:
        add the last char of rotation to the BWT
        add i to positions                            (this is the suffix array)
    RETURN BWT, positions

    """
    BWT = ""
    positions = []
    if not string.endswith("$"):
        string += "$"

    rotations = []
    for i in range(len(string)):
        rotations.append((string[i:] + string[:i], i))   # rotate LEFT: row i starts at position i

    rotations.sort(key=lambda x: x[0])

    for rotation, pos in rotations:
        BWT += rotation[-1]          # last character of the row
        positions.append(pos)        # this list is the suffix array

    return BWT, positions

In [36]:
test = "banana"
print(BWT(test))

('annb$aa', [6, 5, 3, 1, 0, 4, 2])


While this approach works, it's inefficient for large strings. A more efficient implementation uses suffix arrays.

## Suffix Arrays for BWT

Suffix arrays provide a memory-efficient way to compute the BWT. A suffix array contains the starting positions of all suffixes of a string, sorted lexicographically.

$$
\small
\begin{array}{l}
\textbf{function } \text{suffix\_array}(text) \\
\quad \text{Create a list of all suffixes of the text, each paired with its starting position} \\
\quad \text{Sort these suffix-position pairs lexicographically by suffix} \\
\quad \text{Extract just the starting positions from the sorted pairs} \\
\quad \textbf{return } \text{the list of sorted starting positions}
\end{array}
$$


In [37]:
def suffix_array(string: str) -> list[int]:
    """Function to calculate suffix-array for a given string.
    
    Computes the suffix array by sorting all suffixes of the input string
    lexicographically and returning their starting positions.
    
    Args:
        string: The input string to process.
    
    Returns:
        A list of integers representing the starting positions of the
        lexicographically sorted suffixes.
        
    Examples:
        >>> suffix_array('googol')
        [6, 3, 0, 5, 2, 4, 1]
        
        >>> suffix_array('banana$')
        [6, 5, 3, 1, 0, 4, 2]

    Pseudocode:
    FOR i from 0 to length of string - 1:
        store the pair (string[i:], i)                (suffix starting at i)
    SORT the pairs alphabetically by suffix
    RETURN the starting positions, in sorted order

    """
    suffixes = []
    positions = []

    # create suffix array with positions
    for i in range(len(string)):
        # i is index, also number of characters to take off front for suffix
        suffixes.append((string[i:], i))
    
    # lexicographically sort
    suffixes.sort(key=lambda x: x[0])

    # positions of sorted suffixes
    for suffix, pos in suffixes:
        positions.append(pos)
    
    return positions


In [38]:
print(suffix_array('banana$'))

[6, 5, 3, 1, 0, 4, 2]


Once we have the suffix array, we can compute the BWT more efficiently:

$$
\small
\begin{array}{l}
\textbf{function } \text{BWT\_from\_suffix\_array}(text, \text{suffix\_positions}) \\
\quad \text{Initialize an empty string of the same length as the input text} \\
\quad \textbf{for each } \text{position in the suffix positions list do} \\
\quad\quad \textbf{if } \text{the suffix starts at the beginning of the text then} \\
\quad\quad\quad \text{Add the last character of the text to the transformed result} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character preceding the suffix to the transformed result} \\
\quad \textbf{return } \text{the transformed result}
\end{array}
$$

![](assets/full.png)

> Credit: Siren, J. (2016). Burrows-Wheeler Transform for Terabases. In 2016 Data Compression Conference (DCC) (pp. 211–220). 2016 Data Compression Conference (DCC). IEEE. https://doi.org/10.1109/dcc.2016.17 

In [39]:
def BWT_from_suffix_array(
  text: str, 
  suffix_positions: list[int]
  ) -> str:
    """Function to calculate the Burrows-Wheeler Transform from a suffix array.
    
    Computes the Burrows-Wheeler Transform by using the suffix array to identify
    the character that precedes each suffix in the sorted order.
    
    Args:
        text: The input string to transform.
        suffix_positions: The suffix array for the input string, containing the
            starting positions of all suffixes in lexicographical order.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        
    Examples:
        >>> BWT_from_suffix_array("banana$", [6, 5, 3, 1, 0, 4, 2])
        'annb$aa'
        
        >>> BWT_from_suffix_array("googol$", [6, 3, 0, 5, 2, 4, 1])
        'lo$oogg'

    Pseudocode:
    start with an empty list for the last column
    FOR each pos in suffix_positions:
        IF pos == 0:
            add the LAST char of text ($)             (nothing comes before it, so wrap around)
        ELSE:
            add text[pos - 1]                         (the char just before this suffix)
    RETURN the list joined into a string

    """
    last_column = []                        
    for pos in suffix_positions:
        if pos == 0:
            last_column.append(text[-1])      # suffix is the whole text: wrap around to the last char ($)
        else:
            last_column.append(text[pos - 1]) # the char just before this suffix
    return ''.join(last_column)

In [40]:
test = "googol$"
test_positions = [6, 3, 0, 5, 2, 4, 1]
print(BWT_from_suffix_array(test, test_positions))

lo$oogg


---

## FM-Index for Pattern Matching

The BWT becomes particularly powerful when combined with auxiliary data structures to form an FM-Index, which enables efficient pattern matching. Two key components are the count and occurrence arrays.

The FM-index is a compressed full-text index data structure that combines the Burrows-Wheeler Transform (BWT) with auxiliary data structures to enable efficient pattern matching while maintaining a small memory footprint. Developed by Paolo Ferragina and Giovanni Manzini (hence the name "FM"), it's sometimes referred to as "Full-text Minute-space" index.

#### Core Components

The FM-index consists of several key components:

1. **The BWT (L column)** - The last column from the Burrows-Wheeler Matrix, which is a permutation of the original text that groups similar characters together

2. **First column (F)** - Can be represented as an array of |Σ| integers (where Σ is the alphabet), or sometimes not stored at all as it can be derived

3. **Count array (C)** - Stores the number of characters lexicographically smaller than each character in the alphabet

4. **Occurrence table (OT)** - Tracks how many times each character appears up to each position in the BWT

5. **Suffix Array (SA) sample** - A subset of the suffix array positions, often stored at regular intervals to save space

#### Key Operations

The FM-index supports several fundamental operations:

1. **Count** - Returns the number of occurrences of a pattern in the text in O(p) time, where p is the pattern length

2. **Locate** - Returns the positions of all occurrences in the text

3. **Backward search** - The core algorithm that enables efficient pattern matching by traversing the pattern from right to left

4. **LF mapping** - A procedure that maps positions in the L column to corresponding positions in the F column

#### Advantages

The FM-index offers several notable benefits:

1. **Space efficiency** - Occupies space close to the entropy of the indexed text, typically 5nHk(T) + o(n) bits, where Hk(T) is the k-th order entropy

2. **Query performance** - Allows searching for pattern occurrences in O(p + occ log^(1+ε) n) time, where occ is the number of occurrences

3. **Self-indexing** - Encapsulates the indexed data, allowing reconstruction of the original text without storing it separately

4. **Compression** - Takes advantage of the compressibility of the indexed data

5. **Partial decompression** - Only decompresses tiny portions of the compressed file during queries

The FM-index has become particularly important in bioinformatics for efficient DNA and protein sequence alignment, forming the basis of popular tools like Bowtie and BWA that handle massive genomic datasets.

### The implementation

One of the key parts for string matching is to do Last-to-First column mapping (LF mapping) within the BWT matrix. With the LF property, we  need to build two dictionaries for our reference string beforehand:

1. count: e.g.  `{'A': 0, 'C': 2, 'G': 3, 'T': 5}`

Where for each character `a` in a string, `count[a]` contains the number of characters in string that are lexicographically smaller than `a`.

2. occur: `{'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}`

Where for each character `a` in a bwt string, `occur[a][i]` contains the number of occurences of `a` in `bwt_string[0,i], i=1,...,len(bwt_string)` (i.e. the first i characters in bwt string).

With those two dictionaries, we can then start matching the query string to our reference string. 

## Count Array

The count array stores the number of characters lexicographically smaller than each character in the alphabet:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_counts}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary to track character counts} \\
\quad \text{Initialize a result dictionary for cumulative counts} \\
\quad \text{Set initial cumulative count to zero} \\
\quad \textbf{for each } \text{character in the sorted alphabet do} \\
\quad\quad \text{Store the current cumulative count for this character} \\
\quad\quad \text{Increase the cumulative count by the frequency of this character} \\
\quad \textbf{return } \text{the dictionary of cumulative counts}
\end{array}
$$

In [41]:
from collections import Counter

def cal_count(string: str) -> dict[str, int]:
    """Function to count characters lexicographically smaller than each character.
    
    For each character in the alphabet, calculates how many characters in the
    input string are lexicographically smaller than it.
    
    Args:
        string: The input string to analyze.
    
    Returns:
        A dictionary mapping each character to the count of characters
        lexicographically smaller than it.
    
    Examples:
        >>> cal_count('ATGACG')
        {'A': 0, 'C': 2, 'G': 3, 'T': 5}
        
        >>> cal_count('banana')
        {'a': 0, 'b': 3, 'n': 4}

    Pseudocode: 
    count how many times each char appears (freq)
    set cumulative = 0
    FOR each char in sorted order:
        IF char is $:
            skip it                                   (so count of the first letter is 0)
        count[char] = cumulative                      (number of letters smaller than char)
        cumulative += freq[char]
    RETURN count
    """
    freq = Counter(string)              
    count = {}              # {char: how many times it appears}
    cumulative = 0
    for char in sorted(freq):   # alphabetical; '$' sorts first
        if char == '$':
            continue           # skip $ so count['A'] = 0
        count[char] = cumulative     # number of letters smaller than this char
        cumulative += freq[char]
    return count

In [42]:
test = 'ATGACG'
print(cal_count(test))

{'A': 0, 'C': 2, 'G': 3, 'T': 5}


## Occurrence Array

The occurrence array tracks how many times each character appears up to each position in the BWT:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_occurrences}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary mapping each character to an array of zeros} \\
\quad \textbf{for each } \text{position in the transformed text do} \\
\quad\quad \text{Identify the character at the current position} \\
\quad\quad \textbf{for each } \text{character in the alphabet do} \\
\quad\quad\quad \text{Copy the previous occurrence count to the current position} \\
\quad\quad \text{Increment the occurrence count for the current character} \\
\quad \textbf{return } \text{the dictionary of occurrence counts}
\end{array}
$$


In [43]:
def cal_occur(bwt_string: str) -> dict[str, list[int]]:
    """Function to calculate occurrences of each character up to each position.
    
    For each character and each position i, calculates how many times the
    character appears in the substring bwt_string[0:i].
    
    Args:
        bwt_string: The BWT string to analyze.
    
    Returns:
        A dictionary mapping each character to a list of occurrence counts,
        where occur[char][i] is the number of occurrences of char in
        bwt_string[0:i+1].
    
    Examples:
        >>> cal_occur('AG$CG')
        {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}
        
        >>> cal_occur('annb$aa')
        {'$': [0, 0, 0, 0, 1, 1, 1], 'a': [0, 1, 1, 1, 1, 2, 3], 'b': [0, 0, 0, 1, 1, 1, 1], 'n': [0, 0, 2, 2, 2, 2, 2]}

    Pseidocde:
    FOR each distinct char:
        make a list of 0s, one per position
    FOR each position i and its char current:
        FOR each char in the dict:
            IF i > 0:
                copy occur[char][i - 1] into occur[char][i]
        add 1 to occur[current][i]
    RETURN occur                                      (occur[char][i] counts bwt[0..i], inclusive)
    """
    length_bwt = len(bwt_string)
    occur = {char: [0] * length_bwt for char in sorted(set(bwt_string))}   # one list of 0s per char

    for i, current in enumerate(bwt_string):
        for char in occur:
            if i > 0:
                occur[char][i] = occur[char][i - 1]   # copy the previous count forward
        occur[current][i] += 1                        # the char at position i gets +1

    return occur

In [44]:
test = 'AG$CG'
print(cal_occur(test))

{'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}


---

# Pattern Matching with BWT

![](assets/pattern_match.png)

> Credit: ["DNA Sequence Alignment with the Burrows Wheeler Transform"](https://cs.carleton.edu/cs_comps/2324/sequenceAlignment/) from Carleton College

## Alignments using BWT

With the BWT and auxiliary structures in place, we can efficiently find pattern matches using backward search:

$$
\small
\begin{array}{l}
\textbf{function } \text{update\_range}(\text{character}, \text{range\_start}, \text{range\_end}, \text{counts}, \text{occurrences}) \\
\quad \text{Calculate new start position using the character's count and occurrences at range start} \\
\quad \text{Calculate new end position using the character's count and occurrences at range end} \\
\quad \textbf{return } \text{new start and end positions}
\end{array}
$$

In [45]:
def update_range(
  lower: int, 
  upper: int, 
  count: dict[str, int], 
  occur: dict[str, list[int]], 
  a: str) -> tuple[int, int]:
    """Function to update range given character a.
    
    Updates the search range during backward search in the BWT pattern matching
    algorithm when processing character a.
    
    Args:
        lower: The lower boundary of the current range.
        upper: The upper boundary of the current range.
        count: Dictionary mapping each character to the count of lexicographically
            smaller characters.
        occur: Dictionary mapping each character to its occurrence counts at each
            position.
        a: The character being processed in the pattern.
    
    Returns:
        A tuple containing the updated lower and upper boundaries of the range.
    
    Note:
        This function assumes occur[a][-1] = 0 for boundary conditions.

    Pseudocode:
    IF lower == 0:
        a_before_lower = 0
    ELSE:
        a_before_lower = occur[a][lower - 1]          (a's before the range)
    a_through_upper = occur[a][upper]                 (a's through the end of the range)
    lower_new = count[a] + a_before_lower + 1
    upper_new = count[a] + a_through_upper
    RETURN lower_new, upper_new
    """
    # The LF-mapping formula is:
    #   new_lower = count[a] + (number of a's in bwt[0 .. lower-1])
    #   new_upper = count[a] + (number of a's in bwt[0 .. upper]) - 1
    if lower == 0:
        a_before_lower = 0
    else:
        a_before_lower = occur[a][lower - 1]

    # Number of a's in rows 0..upper (inclusive).
    a_through_upper = occur[a][upper]

    new_lower = count[a] + a_before_lower + 1
    new_upper = count[a] + a_through_upper

    return new_lower, new_upper

<div style="text-align: left">
    
$$
\small
\begin{array}{l}
\textbf{function } \text{find\_match}(\text{pattern}, \text{transformed}, \text{counts}, \text{occurrences}, \text{suffix\_positions}) \\
\quad \text{Initialize search range to cover the entire transformed text} \\
\quad \textbf{for each } \text{character in the pattern, processing from right to left do} \\
\quad\quad \text{Update the search range based on the current character} \\
\quad\quad \textbf{if } \text{the range becomes empty then} \\
\quad\quad\quad \textbf{return } \text{empty list as pattern is not found} \\
\quad \text{Collect all suffix positions within the final range} \\
\quad \textbf{return } \text{the list of matching positions}
\end{array}
$$

</div>

In [46]:
def find_match(query: str, reference: str) -> list[int]:
    """Function to find exact matching by applying Burrows-Wheeler Transform.
    
    Searches for all occurrences of the query string within the reference string
    using the Burrows-Wheeler Transform algorithm for efficient pattern matching.
    
    Args:
        query: The pattern string to search for.
        reference: The text string to search within.
    
    Returns:
        A list of integers representing the 0-based starting positions of all
        occurrences of the query string within the reference string. Returns an
        empty list if no matches are found.
    
    Examples:
        >>> find_match('ana', 'banana')
        [1, 3]
        
        >>> find_match('xyz', 'banana')
        []

    Pseudocode:
    IF query is empty:
        RETURN empty list
    IF reference does not end with $:
        append $ to reference
    suffix_positions = suffix_array(reference)
    transformed = BWT_from_suffix_array(reference, suffix_positions)
    count = cal_count(reference)
    occur = cal_occur(transformed)
    set lower = 0, upper = length of transformed - 1  (every row matches the empty string)
    FOR each char in query, from last to first:
        IF char is not in count:
            RETURN empty list                         (char never appears in reference)
        lower, upper = update_range(lower, upper, count, occur, char)
        IF lower > upper:
            RETURN empty list                         (no match)
    collect suffix_positions[row] for every row from lower to upper
    SORT the positions                                (rows are alphabetical, not text order)
    RETURN positions

    """
    
    # Edge case: an empty query has nothing to search for, so return no matches.
    if not query:
        return []

    if not reference.endswith('$'):
        reference = reference + '$'

    suffix_positions = suffix_array(reference)                        # SA
    transformed = BWT_from_suffix_array(reference, suffix_positions)  # BWT
    count = cal_count(reference)       # characters smaller than each char
    occur = cal_occur(transformed)     # running counts in the BWT

    # Every row matches the empty string, so start with all of them.
    lower = 0
    upper = len(transformed) - 1

    
    for char in reversed(query):

        # Edge case: a character that never appears in the reference can't match, and count[char] would raise a KeyError.
        if char not in count:
            return []
        lower, upper = update_range(lower, upper, count, occur, char)
        if lower > upper:
            return []

    matches = []
    for row in range(lower, upper + 1):
        matches.append(suffix_positions[row])

    # The rows are in alphabetical order, not text order. For 'ana' in 'banana' they give [3, 1], so we sort to match the docstring's [1, 3].
    matches.sort()

    return matches

In [47]:
test_query = 'ana'
test = 'banana'
print(find_match(test_query, test))

no_query = 'app'
print(find_match(no_query, test))

[1, 3]
[]


***

## Run-Length Encoding

The Burrows-Wheeler Transform tends to group identical characters into runs, which makes it a natural candidate for run-length encoding (RLE). In RLE, we store each run as a pair: the character and the length of its consecutive occurrence. This is particularly effective when the transformed text contains many long runs.

Conceptually, run-length encoding on the BWT string works as follows:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_encode}(\text{input text}) \\
\quad \text{Start with an empty encoded text} \\
\quad \textbf{if } \text{the input text is empty then} \\
\quad\quad \textbf{return } \text{the empty encoded text} \\
\quad \text{Remember the first character of the input text} \\
\quad \text{Set a counter to one for how many times this character appears in a row} \\
\quad \textbf{for each } \text{next character in the input text} \textbf{ do} \\
\quad\quad \textbf{if } \text{the character is the same as the one being counted} \\
\quad\quad\quad \text{Increase the counter by one} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character being counted and its counter to the encoded text} \\
\quad\quad\quad \text{Begin counting from one for the new character} \\
\quad \text{After the scan, add the final character and its counter to the encoded text} \\
\quad \textbf{return } \text{the encoded text}
\end{array}
$$


In [48]:
def run_length_encode(bwt_string: str) -> str:
    """Run-length encode a BWT string as a compact character-count string.

    Scans the input BWT string from left to right and groups consecutive
    identical characters into runs, returning an encoded string where each
    run is represented as the character followed by its count.

    Args:
    bwt_string: The Burrows-Wheeler transformed string to encode.

    Returns:
    A run-length encoded string of the form 'a3n2b1$1a2', where each
    character is followed by the length of its consecutive run.

    Examples:
    >>> run_length_encode('aaabbc')
    'a3b2c1'

    >>> run_length_encode('annb$aa')
    'a1n2b1$1a2'

    Pseudocde:
    start with empty encoded text
    IF bwt_string is empty:
        RETURN empty encoded text
    current_char = first char, run_length = 1
    FOR each remaining char:
        IF char == current_char:
            run_length += 1
        ELSE:
            add current_char + run_length to encoded text
            current_char = char, run_length = 1
    add the final current_char + run_length to encoded text
    RETURN encoded text


    """
    encoded = ''

    if not bwt_string:
        return encoded
    current_char = bwt_string[0]
    run_length = 1
    for char in bwt_string[1:]:
        if char == current_char:
            run_length += 1
        else:
            encoded += current_char + str(run_length)
            current_char = char
            run_length = 1

    encoded += current_char + str(run_length)
    # Note: this format assumes the input has no digit characters;
    # otherwise a symbol like '3' could be mistaken for a count.
    return encoded

In [49]:
test = 'annb$aa'
print(run_length_encode(test))

a1n2b1$1a2


***

## Run-Length Decoding

Run-length decoding reverses the RLE process by expanding each stored pair back into its original sequence of repeated characters. When combined with the inverse BWT, this allows us to reconstruct the original text from a compressed, run-length encoded representation of the transform.

The pseudocode for decoding is straightforward:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_decode}(\text{encoded text}) \\
\quad \text{Start with an empty decoded text} \\
\quad \text{Begin reading at the start of the encoded text} \\
\quad \textbf{while } \text{there is still unread encoded text} \textbf{ do} \\
\quad\quad \text{Read a single character, which is the symbol to be repeated} \\
\quad\quad \text{Then read the following one or more digits as the number of repeats} \\
\quad\quad \text{Turn this sequence of digits into a number} \\
\quad\quad \text{Add that many copies of the symbol to the decoded text} \\
\quad \textbf{return } \text{the decoded text}
\end{array}
$$


In [50]:
def run_length_decode(encoded: str) -> str:
    """Decode a run-length encoded character-count string.

    Reconstructs the original string by parsing an encoded representation
    where each run is stored as a character followed by its count, and
    expanding each run back into repeated characters.

    Args:
    encoded: A run-length encoded string of the form 'a3n2b1$1a2'.

    Returns:
    The decoded string obtained by expanding all runs in the encoded input.

    Examples:
    >>> run_length_decode('a3b2c1')
    'aaabbc'

    >>> run_length_decode('a1n2b1$1a2')
    'annb$aa'

    Pseudocode:
    start with empty decoded text, i = 0
    WHILE i < length of encoded:
        symbol = encoded[i], move i forward one
        read all following digits into a number
        add symbol repeated that many times to decoded text
    RETURN decoded text

    """
    decoded = ''
    i = 0

   
    while i < len(encoded):
        symbol = encoded[i]
        i += 1
        digits = ''
        while i < len(encoded) and encoded[i].isdigit():
            digits += encoded[i]
            i += 1
        repeat = int(digits)
        decoded += symbol * repeat

    return decoded


In [51]:
test = 'a3b2c12'
print(run_length_decode(test))

aaabbcccccccccccc


## Tricky Parts to Watch Out For

1. **End-of-string character**: Ensure your implementation includes a unique end-of-string character (often '$') that doesn't appear elsewhere in the string. This character should typically be lexicographically smaller than all other characters.

2. **Indexing issues**: Pay careful attention to zero-based versus one-based indexing, especially when implementing the BWT from suffix arrays.

3. **Edge cases**: Handle edge cases properly, such as when the suffix array index is 0, or when dealing with empty strings or patterns.

4. **Performance considerations**: The naive BWT implementation is $O(n^2 \log n)$ due to sorting n strings of length n. Using suffix arrays reduces this to $O(n \log n)$.

5. **Memory usage**: For large strings, storing all rotations can be memory-intensive. The suffix array approach is more memory-efficient.

---

# Summary

The Burrows-Wheeler Transform showcases how a seemingly simple reordering of characters can lead to powerful applications in compression and pattern matching. By understanding both the conceptual foundation and the implementation details, you'll gain insight into one of the most elegant algorithms in computer science, with applications ranging from everyday file compression to cutting-edge genomic research.